## Creacion esquema y tablas

In [0]:
%sql
-- Usar tu catálogo
USE CATALOG bootcamp;
-- Crear esquema Landing
CREATE SCHEMA IF NOT EXISTS bootcamp.landing
COMMENT 'Esquema para archivos';
-- Crear esquema Raw
CREATE SCHEMA IF NOT EXISTS bootcamp.bronze
COMMENT 'Esquema para datos crudos sin procesar';
-- Verificar
SHOW SCHEMAS;

In [0]:
%sql
-- ============================================================
-- PASO 3: Crear the Volume para archivos
-- ============================================================
-- Crear volume tipo MANAGED (Databricks administra el storage)
CREATE VOLUME IF NOT EXISTS bootcamp.landing.archivos
COMMENT 'Volume para almacenar archivos CSV crudos';
-- Verificar que se creó
SHOW VOLUMES IN bootcamp.landing;

In [0]:
%sql
-- ============================================================
-- PASO 4: Verificar que el archivo se subió correctamente
-- ============================================================
-- Listar archivos en el volume
LIST '/Volumes/bootcamp/landing/archivos/';

In [0]:
%sql
-- Leer directo del archivo es posible, y es útil también para analizarlo
SELECT * FROM read_files(
'/Volumes/bootcamp/landing/archivos/properties_raw.csv',
format => 'csv',
header => true
)

In [0]:
%sql
-- ============================================================
-- PASO 5: Crear la tabla properties_bronze
-- ============================================================
-- Primero, eliminamos la tabla si existe (para poder recrearla)
DROP TABLE IF EXISTS bootcamp.bronze.properties_bronze;
-- Crear tabla EXTERNA leyendo el CSV y nos quedamos solo con los registros que tienen url válida
CREATE TABLE bootcamp.bronze.properties_bronze
SELECT * FROM read_files(
'/Volumes/bootcamp/landing/archivos/properties_raw.csv',
format => 'csv',
header => true
)
where url like 'https%' --Filtramos por aquellos links que sean consistentes
;

In [0]:
%sql
-- Verificar que la tabla se creó correctamente
SHOW TABLES IN bootcamp.bronze;

# EDA PARTE 1 - EXPLORACION INICIAL

In [0]:
%sql
-- E1.1
-- CUANTOS REGISTROS CONTIENE LA TABLA bootcamp.bronze.properties_bronze?
select count(*) from bootcamp.bronze.properties_bronze;

In [0]:
%sql
-- E1.2
-- Que columnas posee la tabla?

DESCRIBE bootcamp.bronze.properties_bronze;

In [0]:
%sql

--E1.3
-- Muestra de datos
select
    id,
    ubicacion,
    precio,
    expensas,
    tipo_de_operacion,
    moneda,
    ambientes,
    metros_cuadrados_totales,
    antiguedad,
    estado,
    zona
from bootcamp.bronze.properties_bronze
limit 10;

# EDA PARTE 2 - ANALISIS VALORES NULOS

In [0]:
%sql
-- E2.1
-- Contar nulos por columna

select
    count(*) as total_registros,
    total_registros - count(precio) as total_null_col_precio,
    total_registros - count(expensas) as total_null_col_expensas,
    total_registros - count(tipo_de_operacion) as total_null_tipo_operacion,
    total_registros - count(moneda) as total_null_moneda,
    total_registros - count(ambientes) as total_null_ambientes,
    total_registros - count(metros_cuadrados_totales) as total_null_m2_totales,
    total_registros - count(metros_cuadrados_cubiertos) as total_null_m2_cub,
    total_registros - count(orientacion_cardinal) as total_null_orientacion_cardinal,
    total_registros - count(piso) as total_null_piso,
    total_registros - count(cochera) as total_null_cochera,
    total_registros - count(antiguedad) as total_null_antiguedad,
    total_registros - count(estado) as total_null_estado,
    total_registros - count(zona) as total_null_zona
    from bootcamp.bronze.properties_bronze;
    

In [0]:
-- E2.2 y E2.3
-- Porcentaje de nulos

with porc_nulos as (
    select
    count(*) as cant_reg,
    (cant_reg - count(tr.precio)) *100.0/ cant_reg as porc_null_precio,
    (cant_reg - count(tr.expensas))*100.0/cant_reg as porc_null_expensas,
    (cant_reg - count(tr.ambientes))*100.0/cant_reg as porc_null_ambientes,
    (cant_reg - count(tr.metros_cuadrados_totales))*100.0/cant_reg as porc_null_m2_totales,
    (cant_reg - count(tr.metros_cuadrados_cubiertos))*100.0/cant_reg as porc_null_m2_cub,
    (cant_reg - count(tr.orientacion_cardinal))*100.0/cant_reg as porc_null_orientacion_cardinal, 
    (cant_reg - count(tr.antiguedad))*100.0/cant_reg as porc_null_antiguedad
    from bootcamp.bronze.properties_bronze as tr

)


select
    columna, porcentaje_null
from porc_nulos
UNPIVOT (
    porcentaje_null FOR columna IN (
        porc_null_precio, porc_null_expensas, porc_null_ambientes, 
        porc_null_m2_totales, porc_null_m2_cub,
        porc_null_orientacion_cardinal, porc_null_antiguedad
    )
)
where porcentaje_null>50.0;

# EDA PARTE 3 - CARDINALIDAD Y DISTRIBUCION

In [0]:
%sql
-- E3.1 - Distribución de tipo de operación - ¿Qué valores únicos hay en tipo_de_operacion? ¿Cuántos
-- registros hay de cada tipo? ¿Qué porcentaje representa cada uno? Pista: Usa GROUP BY, COUNT(*), y
-- SUM(COUNT(*)) OVER() para calcular porcentajes
WITH reg as (
    SELECT
        count(*) as total_registros
    from bootcamp.bronze.properties_bronze
)
select
    tipo_de_operacion,
    count(*) as total_registros,
    round(count(*) * 100.0 / sum(count(*)) over(), 4) as porcentaje
    
from bootcamp.bronze.properties_bronze

group by tipo_de_operacion
order by tipo_de_operacion;

In [0]:
%sql

--E3.2 - Distribución de moneda - ¿Qué monedas hay en el dataset? ¿Cuántos registros hay de cada una?
-- ¿Hay alguna moneda que no esperabas? Pista: GROUP BY moneda con conteo y porcentaje


select 
    moneda,
    count(*) as total_registros,
    round(count(*) * 100.0 / sum(count(*)) over(), 4) as porcentaje
from bootcamp.bronze.properties_bronze
group by moneda
order by moneda;

In [0]:
%sql
--E3.3 
-- Distribución de ambientes - ¿Cuántas propiedades hay por cantidad de ambientes? ¿Cuál es la
-- distribución? Ordena por cantidad de ambientes. Pista: GROUP BY ambientes ORDER BY ambientes

select 
    ambientes,
    count(*) as total_registros,
    round(count(*) * 100.0 / sum(count(*)) over(), 4) as porcentaje
from bootcamp.bronze.properties_bronze
group by ambientes
order by ambientes;


In [0]:
%sql
-- E3.4
-- Top zonas - ¿Cuáles son las 15 zonas con más propiedades? Muestra zona, cantidad y porcentaje
-- del total. Pista: GROUP BY zona, COUNT(*), porcentaje, ORDER BY cantidad DESC LIMIT 15

select
    zona,
    count(*) as total_propiedades,
    round(count(*) * 100.0 / sum(count(*)) over(), 4) as porcentaje
from bootcamp.bronze.properties_bronze
group by zona
order by total_propiedades desc;
-- limit 15;

In [0]:
%sql
-- E3.5
-- Distribución de estado - ¿Qué valores hay en la columna estado? ¿Cuántas propiedades hay de
-- cada estado? Pista: GROUP BY estado ORDER BY cantidad DESC

select
    estado,
    count(*) as total_propiedades
from bootcamp.bronze.properties_bronze
group by estado
order by total_propiedades desc;


# EDA PARTE 4 Estadísticas Descriptivas de Variables Numéricas

## Creacion Vista temporal con campos convertidos a double

In [0]:
%sql
create or replace temporary View propiedades_clean as
select
CASE
  WHEN precio RLIKE '^[^a-zA-Z]+$' THEN precio::double
  ELSE NULL
  END as precio,
  moneda,
CASE
  WHEN ambientes RLIKE '^[^a-zA-Z]+$' THEN ambientes::double
  ELSE NULL
END as ambientes
 ,
 CASE
  WHEN metros_cuadrados_totales RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_totales::double
  ELSE NULL
END as metros_cuadrados_totales
 ,
 CASE
  WHEN metros_cuadrados_cubiertos RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_cubiertos::double
  ELSE NULL
END as metros_cuadrados_cubiertos
 ,
 CASE
  WHEN antiguedad RLIKE '^[^a-zA-Z]+$' THEN antiguedad::double
  ELSE NULL
END as antiguedad,
tipo_de_operacion
,id
,ubicacion
,numero
,calle
,expensas
,orientacion_cardinal
,orientacion_inmueble
,piso
,cochera
,estado
,tipo_vendedor
,url
,zona
,fecha
,hora
from bootcamp.bronze.properties_bronze

In [0]:
%sql
describe propiedades_clean;
--select * from propiedades_clean;

## CREACION TABLA VISTA TEMPORAL 2

In [0]:
%sql
-- ============================================================
-- Creamos la nueva vista temporal
-- ============================================================
CREATE OR REPLACE TEMPORARY VIEW propiedades_clean_2 AS
(
    WITH limites AS (
    SELECT 
      moneda,
      tipo_de_operacion,
      PERCENTILE(precio, 0.01) AS p01,
      PERCENTILE(precio, 0.99) AS p99
    FROM propiedades_clean
    WHERE precio > 0
      AND moneda IN ('USD', 'ARS')
      AND tipo_de_operacion IN ('venta', 'alquiler')
    GROUP BY moneda, tipo_de_operacion
  )
  SELECT p.*
  FROM propiedades_clean p
  JOIN limites l 
    ON p.moneda = l.moneda 
    AND p.tipo_de_operacion = l.tipo_de_operacion
  WHERE p.precio BETWEEN l.p01 AND l.p99
);

In [0]:
%sql

select * from propiedades_clean_2 limit 10;

## E4.1 - Estadísticas de precio por moneda y tipo de operación

In [0]:
%sql

select
    moneda,
    tipo_de_operacion as operacion,
    count(*) as cantidad_registros,
    min(precio) as precio_min,
    max(precio) as precio_max,
    median(precio) as precio_medio,
    percentile(precio,0.25) as precio_p_25,
    percentile(precio,0.75) as precio_p_57
from propiedades_clean_2
where precio > 0
group by moneda, tipo_de_operacion
order by cantidad_registros desc;

## E4.2 - Estadísticas de metros cuadrados

In [0]:
%sql

with m2_totales as (
    select
        "m2_totales" as columna,
        count(*) as cantidad_registros,
        min(metros_cuadrados_totales) as min_,
        max(metros_cuadrados_totales) as max,
        avg(metros_cuadrados_totales) as avg,
        median(metros_cuadrados_totales) as median
    from propiedades_clean_2
    where metros_cuadrados_totales is not null and metros_cuadrados_totales > 0
    
),
m2_cubiertos as (
    select
        "m2_cubiertos" as columna,
        count(*) as cantidad_registros,
        min(metros_cuadrados_cubiertos) as min,
        max(metros_cuadrados_cubiertos) as max,
        avg(metros_cuadrados_cubiertos) as avg,
        --median(metros_cuadrados_cubiertos) as median_m2_cubiertos
        percentile(metros_cuadrados_cubiertos,0.5) as median
    from propiedades_clean_2
    where metros_cuadrados_cubiertos is not null and metros_cuadrados_cubiertos > 0
    
)


select 
    mt.*  
from m2_totales mt

union all

select 
    *
from m2_cubiertos;

## E4.3 - Análisis de antigüedad

In [0]:
%sql
with total_registros as (
    select
        count(*) as total
    from propiedades_clean_2
)

select 
    antiguedad,
    count(*) as registros,
    tr.total as total_registros,
    round((registros/total_registros)*100.0,4) as porcentaje    
from propiedades_clean_2 
cross join total_registros tr
group by antiguedad, total_registros
order by registros desc;

## E4.4 - Estadísticas de antigüedad sin placeholders

In [0]:
%sql

with antiguedad_limpia as (
    select 
    *
    from propiedades_clean_2
    where 
        antiguedad is not null
        and antiguedad > 0
        and antiguedad !="999"
)
-- select
--     count(*)
-- from propiedades_clean_2;

-- select
--     count(*)
-- from antiguedad_limpia;

select    
    count(*) as total_registros,
    min(antiguedad) as antiguedad_min,
    max(antiguedad) as antiguedad_max,
    avg(antiguedad) as antiguedad_avg,
    median(antiguedad) as antiguedad_media
from antiguedad_limpia;


#EDA PARTE 5: Detección de Problemas de Calidad

## E5.1 - Reporte de calidad usando CTEs

In [0]:
%sql

with total as(
    select
        count(*) as total_reg        
    from propiedades_clean_2
),
problemas as (
    select
        count(case when precio is null or precio <=0 then 1 end) as precios_inv,
        count(case when metros_cuadrados_totales is null or metros_cuadrados_totales <=0 then 1 end) as m2_inv,
        count(case when antiguedad == 999 or antiguedad =='999' then 1 end) as antiguedad_inv,
        count(case when ambientes is null or ambientes <=0 then 1 end) as ambientes_inv,
        count(case when moneda is null or moneda ==""then 1 end) as moneda_inv
    from propiedades_clean_2
)

select
 t.total_reg,
 p.precios_inv,
 (p.precios_inv/t.total_reg)*100.0 as porc_precios_inv,
 p.m2_inv,
 (p.m2_inv/t.total_reg)*100.0 as porc_m2_inv,
 p.antiguedad_inv,
 (p.antiguedad_inv/t.total_reg)*100.0 as porc_antiguedad_inv,
 p.ambientes_inv,
 (p.ambientes_inv/t.total_reg)*100.0 as porc_ambientes_inv,
 p.moneda_inv,
 (p.moneda_inv/t.total_reg)*100.0 as porc_moneda_inv
from total t
cross join problemas p;


##E5.2-Detectar duplicados

In [0]:
%sql

with grupos as (
    select
        precio,
        url,
        count(*) as cantidad_reg
    from bootcamp.bronze.properties_bronze
    group by precio, url
    having cantidad_reg > 1 
)


select
    count(*) as cant_dup,
    sum(cantidad_reg) as total_reg_duplicados,
    sum(cantidad_reg - 1) as excedente_por_reg
from grupos;



##E5.3 - Ver ejemplos de duplicados

In [0]:
%sql

with duplicados as (
    select
        --id,
        precio,
        url,
        count(*) as cant_dup
    from bootcamp.bronze.properties_bronze
    group by precio, url
    having cant_dup > 1
)

select
    *    
from duplicados
order by cant_dup desc
limit 10;

##E5.4 - Detectar outliers en precio

In [0]:
%sql

with stats as(
    select
        moneda,
        percentile(precio,0.01) as p1,
        percentile(precio,0.99) as p99
    from propiedades_clean_2
    where precio>0
    group by moneda
)
-- moneda_group as (
--     select
--         moneda,
--         count(*) as total_reg        
--     from propiedades_clean_2
--     where moneda =="ARS"    
-- )

select
    p.moneda,
    "muy_bajo_p1" as outlier,
    count(*) as cantidad
from propiedades_clean_2 p
inner join stats s on s.moneda == p.moneda
where p.precio < s.p1 and p.precio>0
group by p.moneda

union all

select 
    p.moneda,
    "muy_alto_p99" as outlier,
    count(*) as cantidad
from propiedades_clean_2 p
inner join stats s on s.moneda == p.moneda
where p.precio > s.p99
group by p.moneda
order by moneda, outlier

In [0]:
%sql

select zona, count(*) cant_reg from propiedades_clean_2 group by zona order by cant_reg desc;

#PARTE 6 ANALISIS AVANZADO Y DOCUMENTACION

## E6.1 - Ranking de zonas por precio

In [0]:
%sql

select
    row_number() over(order by avg(precio) desc) as rn,
    zona,
    "ARS" as moneda,
    round(AVG(precio),4) as precio_promedio,
    count(*) as cant_prop
from propiedades_clean_2
where precio>0 and moneda =="ARS" and tipo_de_operacion =="alquiler"
group by zona


##E6.2-Comparar con promedio general

In [0]:
%sql

with p_zona as (
    select
     zona,
     avg(precio) as avgz
    from propiedades_clean_2
    where moneda =="ARS"
    group by zona
)

select
    zona,
    avgz as avg_zona,    
    avg(avgz) over () as avgg
from p_zona